In [1]:
import sys
print(sys.executable)

/Users/mh/opt/anaconda3/envs/ai-train-2025/bin/python


In [2]:
!{sys.executable} -m pip install -U datasets huggingface_hub

In [3]:
import datasets, transformers, torch
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

datasets: 4.2.0
transformers: 4.56.1
torch: 2.5.1


In [4]:
%pip install -q -U datasets accelerate peft safetensors  # 既に入っていれば数秒で終わります

import torch, platform
USE_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
DEVICE = "mps" if USE_MPS else "cpu"
print("python:", platform.python_version(), "| torch:", torch.__version__, "| device:", DEVICE)

Note: you may need to restart the kernel to use updated packages.
python: 3.11.13 | torch: 2.5.1 | device: mps


In [5]:
from datasets import load_dataset, Dataset

try:
    ds = load_dataset("yahma/alpaca-cleaned", split="train[:200]")
    print(ds)
    print(ds[0])
except Exception as e:
    print("fallback due to:", e)
    samples=[{"instruction":"Say hello","input":"","output":"Hello!"},
             {"instruction":"Add two numbers","input":"2 and 3","output":"5"},
             {"instruction":"和訳","input":"machine learning","output":"機械学習"},
             {"instruction":"分類","input":"This is great.","output":"positive"},
             {"instruction":"Capital of Japan?","input":"","output":"Tokyo"},
             {"instruction":"Summarize","input":"PyTorch is a DL framework.","output":"It is a deep learning framework."},
             {"instruction":"英訳","input":"大規模言語モデル","output":"large language model"},
             {"instruction":"List 2 fruits","input":"","output":"Apple, Banana"}]
    ds = Dataset.from_list(samples)
    print(ds[0])

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 200
})
{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = "gpt2"   # safetensors / 軽量
tok = AutoTokenizer.from_pretrained(model_id)

def to_prompt(ex):
    text = (f"### Instruction:\n{ex['instruction']}\n"
            f"### Input:\n{ex['input']}\n"
            f"### Response:\n{ex['output']}\n")
    return {"prompt": text}

ds_p = ds.map(to_prompt)
print(ds_p[0]["prompt"][:200].replace("\n","⏎"))
print("chars:", len(ds_p[0]["prompt"]))
print("chars:", len(ds_p[1]["prompt"]))

### Instruction:⏎Give three tips for staying healthy.⏎### Input:⏎⏎### Response:⏎1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean pr
chars: 835
chars: 388


In [7]:
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

BLOCK = 256  # MPSでも安定。重ければ 192 へ
def tokenize(ex):
    text = ex["prompt"] + tok.eos_token
    enc = tok(text, max_length=BLOCK, truncation=True, padding="max_length")
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}

keep_cols = [c for c in ds_p.column_names]
ds_tok = ds_p.map(tokenize, remove_columns=keep_cols)
#tok = AutoTokenizer.from_pretrained(model_id)

ids0 = ds_tok[0]["input_ids"]; am0 = ds_tok[0]["attention_mask"]
print("len(ids)=", len(ids0), "| sum(mask)=", sum(am0))
print("tail decode:", tok.decode(ids0[-80:]))

len(ids)= 256 | sum(mask)= 173
tail decode: <|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endof

In [9]:
def add_labels(ex):
    ex["labels"] = [t if m==1 else -100 for t,m in zip(ex["input_ids"], ex["attention_mask"])]
    #(t, m) = ("input_ids", "attention_mask")
    return ex

ds_lab = ds_tok.map(add_labels)
print("labels tail (20):", ds_lab[0]["labels"][-20:])

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

labels tail (20): [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [8]:
def add_labels(ex):
    ex["labels"] = [t if m==1 else -100 for t,m in zip(ex["input_ids"], ex["attention_mask"])]
    return ex

ds_lab = ds_tok.map(add_labels)
print("labels tail (20):", ds_lab[0]["labels"][-20:])

labels tail (20): [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [11]:
def pack(dataset, block=BLOCK):
    ids=[]; mask=[]; labels=[]
    for r in dataset:
        ids += r["input_ids"]; mask += r["attention_mask"]; labels += r["labels"]
        #ids    = [101, 102, 201, 202, 301, 302]
        #mask   = [1, 1, 1, 1, 1, 1]
        #labels = [101, 102, 201, 202, 301, 302]
    n = len(ids)//block
    out=[]
    for i in range(n):
        s=i*block; e=s+block
        out.append({"input_ids":ids[s:e],"attention_mask":mask[s:e],"labels":labels[s:e]})
    return out

packed = pack(ds_lab, BLOCK)
print("packed batches:", len(packed), "| each len:", len(packed[0]["input_ids"]))
print("packed[0] head decode:", tok.decode(packed[0]["input_ids"][:80]))

packed batches: 200 | each len: 256
packed[0] head decode: ### Instruction:
Give three tips for staying healthy.
### Input:

### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2.


In [18]:
import torch
from torch.utils.data import Dataset, DataLoader

#PackDS：行（辞書の各リスト）→ テンソル [T]
class PackDS(Dataset):
    def __init__(self, rows): self.rows = rows
    #__len__() が無いと、DataLoader がデータのサイズを認識できません
    def __len__(self): return len(self.rows)
    #__getitem__(i) が無いと、DataLoader が i 番目のデータを取り出せません
    def __getitem__(self, i):
        r = self.rows[i]
        return {k: torch.tensor(v, dtype=torch.long) for k,v in r.items()}

#packed = pack(ds_lab, BLOCK)
#	•	DataLoader：行を束ねて → バッチ [B, T]
dl = DataLoader(PackDS(packed), batch_size=2, shuffle=True)
sほwh
for k,v in batch.items(): print(k, v.shape, v.dtype)

input_ids torch.Size([2, 256]) torch.int64
attention_mask torch.Size([2, 256]) torch.int64
labels torch.Size([2, 256]) torch.int64


In [15]:
import torch
from torch.utils.data import Dataset, DataLoader

class PackDS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {k: torch.tensor(v, dtype=torch.long) for k,v in r.items()}

dl = DataLoader(PackDS(packed), batch_size=2, shuffle=True)
batch = next(iter(dl))
for k,v in batch.items(): print(k, v.shape, v.dtype)

input_ids torch.Size([2, 256]) torch.int64
attention_mask torch.Size([2, 256]) torch.int64
labels torch.Size([2, 256]) torch.int64


In [16]:
model = AutoModelForCausalLM.from_pretrained(model_id).to(DEVICE)
model.config.pad_token_id = tok.pad_token_id

batch = {k: v.to(DEVICE) for k,v in batch.items()}
with torch.no_grad():
    loss_before = float(model(**batch).loss)
print("loss_before:", round(loss_before,4))

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


loss_before: 2.8321


In [17]:
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
opt.zero_grad()
loss = model(**batch).loss
loss.backward()
# （参考）lm_head 勾配ノルムを覗く
with torch.no_grad():
    norms = []
    for n,p in model.named_parameters():
        if p.grad is not None and "lm_head" in n:
            norms.append((n, float(p.grad.norm().detach().cpu())))
opt.step()

with torch.no_grad():
    loss_after = float(model(**batch).loss)
print("loss after 1 step:", round(loss_after,4))
print("grad norms (sample):", norms[:3])

loss after 1 step: 2.1784
grad norms (sample): []


In [ ]:
import os, json
BASE = os.path.expanduser("~/Projects/llm-train/token_pack_train_v0_2")  # 好みで変更OK
os.makedirs(BASE, exist_ok=True)
with open(f"{BASE}/notes.txt","w") as f:
    f.write(f"device={DEVICE}, block={BLOCK}, packed={len(packed)}\n")
    f.write(f"loss_before={loss_before}, loss_after={loss_after}\n")
print("saved:", BASE)